# Time Evolution of two weakly coupled spin-1/2
This notebook uses the Lindblad formalism to simulate the time evolution of two weakly exchange coupled spin-1/2. It includes:
- Spin Hamiltonian
- Constructing collapse operators
- Time Evolution simulation
- Fourier transform
- Tip-field dependence 

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

from spinfinity import lind, ESR, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Spin Hamiltonian

In [ ]:
# Initailze SpinSystem
spinsys = SpinSys([0.5, 0.5])

spinsys.B = [0,0.0,0.5] # T
spinsys.BTip = [1e-9, 0.0, 0.0] # T
spinsys.T = 1 # K

# Coupling
J_GHz = 0.02
spinsys.Jvector[0, :] = np.array([J_GHz*spinsys.GHztomeVConversion, J_GHz*spinsys.GHztomeVConversion, J_GHz*spinsys.GHztomeVConversion])
spinsys.DipoleBool = [[False]]

# Define the Tunnel paramters
spinsys.TipPolarization = [0.0,0.0,0.5]
spinsys.U = [0, 0]
spinsys.G_ss = [0.5e-9, 0.5e-9]
spinsys.G_st = 0.05e-9
spinsys.G_tt = 1
spinsys.V_DC = 0 # mV

hamil.calc_EigEnergies(spinsys)
hamil.calc_EigStates(spinsys)
bath.calc_Rates(spinsys)
bath.calc_Populations(spinsys)
bath.calc_Lifetimes(spinsys)

# Visualization
hamil.plot_EigenMatrix(spinsys);
freq_array = np.linspace(13.7, 14.3, 1000)
ESR.plot_FreqSweep(spinsys, freq_array=freq_array, lw=0.005);

#print(f'EigenStates: {spinsys.eigVectors}')

In [ ]:
c_ops = lind.calc_collapseOperators(spinsys);
spinsys.collapse_ops = c_ops

psi_exc = Qobj([0,0,1,0])  # initial state for the time evolution
time_array = np.linspace(0, 2e3, 1001)  # time array for the time evolution
lind.calc_TimeEvolution(spinsys, psi_0=psi_exc, time_array=time_array, spin_idx_array=[0])   # calculate the time evolution for a specific time range and initial state. 
lind.plot_TimeEvolution(spinsys);   # plot the calculated time evolution

# # Density matrix evolution
lind.plot_DensityMatrix(spinsys, "index", time_index=[0,-1]);

## Fourier transform

In [ ]:
rd = lind.plot_FourierFreqs(spinsys)
print(rd['frequency_peaks'])


## Tip-field dependence

In [ ]:
# time array for the time evolution
time_array = np.linspace(0, 1e3, 1000) # ns
BTip_array = np.linspace(-3e-3, 3e-3, 301)  # Tesla

# Caluclation
result = lind.calc_TimeEvolution_Tipfield(spinsys, psi_0=psi_exc, time_array=time_array, BTip_array=BTip_array);

# Plot
lind.plot_TimeEvolution_Tipfield(spinsys);
